In [12]:
'''

This script analyses the rotational curves from SPARC and MP21 catalogues.

Written by Anton Rudakovskyi, BITP NASU.

License: cc by-sa 4.0

'''

In [12]:
import glob
import time
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
import sys
from io import StringIO

import os
import pickle
import dynesty
from dynesty import plotting as dyplot
from scipy.stats import gaussian_kde
from scipy.optimize import fsolve, minimize, broyden1
import galpynamics.dynamic_component as dc

from likelihoodgenerator import LikelihoodGenerator, gauss
from models_sparc import H0, rho_crit

import warnings
warnings.filterwarnings("ignore")

In [13]:
h = H0*10

In [13]:
def create_folders(models):

    for model in models:
        try:
            os.mkdir(model)
        except FileExistsError:
            pass

        try:
            os.mkdir('{}/corner_plot'.format(model))
        except FileExistsError:
            pass
        
        
def m200(c200, v200):
    
    r200 = v200 / (10 * H0)
    
    m200 = (4*np.pi/3) * 200 * (rho_crit * 1e9) * r200**3
    
    return m200



def logc200_m200(log10m200):
    '''C200-M200 based on the Maccio et al. 2008 '''
    return 0.830 - 0.098 * (log10m200 + np.log10(h) - 12)
        

    

def lp_nfw_cdm(theta, mlums, di):
    '''NFW  C200-M200 and M*/M200 relations (following Li et al. 2020 ) '''
    c200 = theta[1]
    logc200 = np.log10(c200)
    v200 = theta[0]
    d = theta[2]
    M_200 = m200(c200, v200)
    log10m200 = np.log10(M_200)
 
    logc200mean = logc200_m200(log10m200)
   
    Y = np.array(theta[4:])
    mstar = sum(Y*mlums[1:]) * (d/di)**2
    Xstar = np.log10(mstar) - log10m200 #like in Allaert-17
    
    Xstar_mean = np.log10((2 * 0.0351) * ((M_200/10**11.59)**(-1.376) + (M_200/10**11.59)**0.608)**(-1))
    
    return gauss(logc200,logc200mean,0.11) + gauss(Xstar,Xstar_mean,0.15)

    #return gauss(logc200,logc200mean,0.11)


def lp_dc14_cdm(theta, mlums, di):
    
    ''' DC14 C200-M200 and M*/M200 relations (following Li et al. 2020 with corrections)
'''
    c200 = theta[1]
    logc200 = np.log10(c200)
    v200 = theta[0]
    d = theta[2]
    M_200 = m200(c200, v200)
    log10m200 = np.log10(M_200)
    logc200mean_nfw = logc200_m200(log10m200)
    Y = np.array(theta[4:])
    mstar = sum(Y*mlums[1:]) * (d/di)**2
    Xstar = np.log10(mstar) - log10m200 #like in Allaert-17
    logc200mean = logc200mean_nfw + np.log10(1 + 0.00003 * np.exp(3.4* (Xstar+4.5) )) # according to the original DC14 paper. 
    #Warning: we reproduce Li et al. 2020  with this relation only
    
    Xstar_mean = np.log10((2 * 0.0351) * ((M_200/10**11.59)**(-1.376) + (M_200/10**11.59)**0.608)**(-1))
    return gauss(logc200,logc200mean,0.11) + gauss(Xstar,Xstar_mean,0.15)
    

In [14]:
save = True
plot = True

processes = 6
nlive = 500


galaxy_list = ['IC2574']

model_prior = 'DC14_cdm' # cdm means priors similar to the Li et al. 2020

models = ['DC14']



if __name__ == "__main__":

    for model in models:
        
        
        create_folders(models)
        
        for galaxy in galaxy_list:

            file_name = galaxy


            LGenerator = LikelihoodGenerator()
            
            LGenerator.initialize_galaxy(galaxy)
            
            mlums = LGenerator.mlums
            di = LGenerator.additional_parameters[0]

            LGenerator.set_model(model, lum_model='full', mlums = mlums, di = di)
            
            
            ll = LGenerator.log_likelihood
            path_to_res = model
            
            if model_prior == 'NFW_cdm' and model=='NFW':
                
                def ll(theta):
                    return LGenerator.log_likelihood(theta) + lp_nfw_cdm(theta, mlums, di)
                
                path_to_res = model_prior
                
                create_folders([model_prior])
        
            if model_prior == 'DC14_cdm' and model=='DC14':
                
                def ll(theta, mlums=mlums, di=di):
                    return LGenerator.log_likelihood(theta) + lp_dc14_cdm(theta, mlums, di)
                
                path_to_res = model_prior
                
                create_folders([model_prior])
        

            with Pool(processes=processes) as pool:
                try:
                    #dsampler = dynesty.NestedSampler(ll, LGenerator.ptform, ndim = LGenerator.ndim, nlive = nlive, pool = pool, queue_size = processes)
                    dsampler = dynesty.DynamicNestedSampler(ll, LGenerator.ptform, ndim = LGenerator.ndim, nlive = nlive, pool = pool, queue_size = processes)

                    dsampler.run_nested(maxiter_init=30000, maxiter_batch=10000, maxbatch=10, print_progress = True)
                    
                    #dsampler.run_nested(maxiter=30000, print_progress = True)
                    
#                except TimeoutError:
                except Exception as e:
                    file_name += '_fail'
                    print(e)

            dresults = dsampler.results
            logz = dresults.get(['logz'][-1])[-1]
            logzerr = dresults.get(['logzerr'][-1])[-1]

            with open(path_to_res + '/results.txt', 'a') as fout:
                fout.write('{} \t {:.3f} \t {:.3f} \n'.format(galaxy, logz, logzerr))


            save = True
            plot = True

            if save:

                fout='{}/dresults_{}.pkl'.format(model, file_name)

                with open(fout, 'wb') as fn:
                    pickle.dump(dresults, fn)

            #                 fout_dsampler='{}/dsampler_{}_{}.pkl'.format(model, file_name, lum_model)

            #                 with open(fout_dsampler, 'wb') as fn:
            #                     pickle.dump(dsampler, fn)
            if plot:          
                fig, ax = dyplot.cornerplot(dresults, labels=LGenerator.parameters, color='blue', show_titles=True, smooth = 0.05)  
                fig.savefig('{}/corner_plot/{}.png'.format(path_to_res, file_name))
                plt.close() 
                
                

8616it [05:55, 24.22it/s, +500 | bound: 32 | nc: 1 | ncall: 41458 | eff(%): 21.989 | loglstar:   -inf < -88.287 <    inf | logz: -104.896 +/-  1.472 | dlogz:  0.001 >  0.509]     
